In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from torch_geometric.data import HeteroData
import pickle
import os
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.loader import NeighborLoader, LinkNeighborLoader
from torch_geometric.utils import negative_sampling
from torch_geometric.transforms import AddSelfLoops, ToUndirected
with open('grph.pickle', 'rb') as f:
    G = pickle.load(f)

In [2]:
chemicals = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'chemical'])
diseases = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'disease'])
genes = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'gene'])

chem_idx = {c: i for i, c in enumerate(chemicals)}
disease_idx = {d: i for i, d in enumerate(diseases)}
gene_idx = {g: i for i, g in enumerate(genes)}

print(chem_idx)
print(disease_idx)
print(gene_idx)

# Extract node embeddings
chem_features = np.array([G.nodes[c]['embedding'] for c in chemicals], dtype=np.float32)
disease_features = np.array([G.nodes[d]['embedding'] for d in diseases], dtype=np.float32)
gene_features = np.array([G.nodes[g]['embedding'] for g in genes], dtype=np.float32)

# Extract edges
chem_gene_edges = []
chem_disease_edges = []
gene_disease_edges = []
for u, v, data in G.edges(data=True):
    edge_type = data['edge_type']
    if edge_type == 'chem_gene':
        chem_gene_edges.append([chem_idx[u], gene_idx[v]])
    elif edge_type == 'chem_disease':
        chem_disease_edges.append([chem_idx[u], disease_idx[v]])
    elif edge_type == 'gene_disease':
        gene_disease_edges.append([gene_idx[u], disease_idx[v]])


# Convert to tensors
chem_gene_edges = torch.tensor(chem_gene_edges, dtype=torch.long).t()
chem_disease_edges = torch.tensor(chem_disease_edges, dtype=torch.long).t()
gene_disease_edges = torch.tensor(gene_disease_edges, dtype=torch.long).t()


{'C000015': 0, 'C000025': 1, 'C000050': 2, 'C000061': 3, 'C000081': 4, 'C000090': 5, 'C000109': 6, 'C000121': 7, 'C000123': 8, 'C000152': 9, 'C000154': 10, 'C000188': 11, 'C000228': 12, 'C000297': 13, 'C000298': 14, 'C000380': 15, 'C000433': 16, 'C000449': 17, 'C000470': 18, 'C000474': 19, 'C000475': 20, 'C000481': 21, 'C000482': 22, 'C000483': 23, 'C000488': 24, 'C000490': 25, 'C000499': 26, 'C000505': 27, 'C000511': 28, 'C000515': 29, 'C000516': 30, 'C000518': 31, 'C000520': 32, 'C000536': 33, 'C000541': 34, 'C000543': 35, 'C000588486': 36, 'C000588503': 37, 'C000588556': 38, 'C000588561': 39, 'C000588582': 40, 'C000588664': 41, 'C000588695': 42, 'C000588741': 43, 'C000588809': 44, 'C000588914': 45, 'C000588915': 46, 'C000588918': 47, 'C000588958': 48, 'C000589002': 49, 'C000589029': 50, 'C000589030': 51, 'C000589147': 52, 'C000589162': 53, 'C000589248': 54, 'C000589253': 55, 'C000589285': 56, 'C000589286': 57, 'C000589288': 58, 'C000589290': 59, 'C000589451': 60, 'C000589472': 61, '

In [3]:
print(chem_features.shape)
print(disease_features.shape)
print(gene_features.shape)

max_dim = max(chem_features.shape[1], disease_features.shape[1], gene_features.shape[1])
chem_features = np.pad(chem_features, ((0, 0), (0, max_dim - chem_features.shape[1])), mode='constant')
disease_features = np.pad(disease_features, ((0, 0), (0, max_dim - disease_features.shape[1])), mode='constant')
gene_features = np.pad(gene_features, ((0, 0), (0, max_dim - gene_features.shape[1])), mode='constant')


print(chem_features.shape)
print(disease_features.shape)
print(gene_features.shape)

(17795, 2217)
(7296, 2377)
(28680, 2363)
(17795, 2377)
(7296, 2377)
(28680, 2377)


In [4]:
data = HeteroData()
data['chemical'].x = torch.tensor(chem_features, dtype=torch.float)
data['disease'].x = torch.tensor(disease_features, dtype=torch.float)
data['gene'].x = torch.tensor(gene_features, dtype=torch.float)
data['chemical', 'chem_gene', 'gene'].edge_index = chem_gene_edges
data['chemical', 'chem_disease', 'disease'].edge_index = chem_disease_edges
data['gene', 'gene_disease', 'disease'].edge_index = gene_disease_edges
print(data)

data = ToUndirected()(data)
data = AddSelfLoops()(data)
print(data)

HeteroData(
  chemical={ x=[17795, 2377] },
  disease={ x=[7296, 2377] },
  gene={ x=[28680, 2377] },
  (chemical, chem_gene, gene)={ edge_index=[2, 812241] },
  (chemical, chem_disease, disease)={ edge_index=[2, 3467034] },
  (gene, gene_disease, disease)={ edge_index=[2, 34221] }
)
HeteroData(
  chemical={ x=[17795, 2377] },
  disease={ x=[7296, 2377] },
  gene={ x=[28680, 2377] },
  (chemical, chem_gene, gene)={ edge_index=[2, 812241] },
  (chemical, chem_disease, disease)={ edge_index=[2, 3467034] },
  (gene, gene_disease, disease)={ edge_index=[2, 34221] },
  (gene, rev_chem_gene, chemical)={ edge_index=[2, 812241] },
  (disease, rev_chem_disease, chemical)={ edge_index=[2, 3467034] },
  (disease, rev_gene_disease, gene)={ edge_index=[2, 34221] }
)


In [5]:
data['gene'].num_nodes - torch.unique(data['gene', 'gene_disease', 'disease'].edge_index[0]).size(0)

19568

In [6]:
if isinstance(data, HeteroData):
    for edge_type in data.edge_types:
        if 'edge_index' in data[edge_type]:
            data[edge_type].edge_index = data[edge_type].edge_index.contiguous()

# If 'data' is a homogeneous Data object (for completeness, though your code suggests hetero)
else:
    data.edge_index = data.edge_index.contiguous()

In [7]:
class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, metadata):
        super().__init__()
        # Define HeteroConv for each edge type
        self.conv1 = HeteroConv({
            ('chemical', 'chem_gene', 'gene'): SAGEConv((-1, -1), hidden_channels),
            ('chemical', 'chem_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
            ('gene', 'gene_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
            ('gene', 'rev_chem_gene', 'chemical'): SAGEConv((-1, -1), hidden_channels),
            ('disease', 'rev_chem_disease', 'chemical'): SAGEConv((-1, -1), hidden_channels),
            ('disease', 'rev_gene_disease', 'gene'): SAGEConv((-1, -1), hidden_channels)
        }, aggr='sum')
        self.conv2 = HeteroConv({
            ('chemical', 'chem_gene', 'gene'): SAGEConv(hidden_channels, hidden_channels),
            ('chemical', 'chem_disease', 'disease'): SAGEConv(hidden_channels, hidden_channels),
            ('gene', 'gene_disease', 'disease'): SAGEConv(hidden_channels, hidden_channels),
            ('gene', 'rev_chem_gene', 'chemical'): SAGEConv(hidden_channels, hidden_channels),
            ('disease', 'rev_chem_disease', 'chemical'): SAGEConv(hidden_channels, hidden_channels),
            ('disease', 'rev_gene_disease', 'gene'): SAGEConv(hidden_channels, hidden_channels)
        }, aggr='sum')
    
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: x.relu() for key, x in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict


In [8]:
from torch_geometric.loader import LinkNeighborLoader

def predict_edges(drug_emb, disease_emb, edge_index):
    row, col = edge_index
    return (drug_emb[row] * disease_emb[col]).sum(dim=-1)

def train(model, data, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0
    
    # Use LinkNeighborLoader for chemical-disease edges
    loader = LinkNeighborLoader(
        data,
        num_neighbors=[30, 10],
        batch_size=128,
        edge_label_index=(('chemical', 'chem_disease', 'disease'), data['chemical', 'chem_disease', 'disease'].edge_index),
        shuffle=True
    )
    
    for batch in tqdm(loader):
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        out = model(batch.x_dict, batch.edge_index_dict)
        drug_emb = out['chemical']
        disease_emb = out['disease']
        
        # Positive edges
        pos_edge_index = batch['chemical', 'chem_disease', 'disease'].edge_index
        if pos_edge_index.size(1) == 0:
            continue
        pos_score = predict_edges(drug_emb, disease_emb, pos_edge_index)
        
        # Negative edges
        neg_edge_index = negative_sampling(
            edge_index=pos_edge_index,
            num_nodes=(batch['chemical'].num_nodes, batch['disease'].num_nodes),
            num_neg_samples=pos_edge_index.size(1)
        )
        neg_score = predict_edges(drug_emb, disease_emb, neg_edge_index)
        
        # Loss
        scores = torch.cat([pos_score, neg_score])
        labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).to(device)
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else float('inf')


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HeteroGNN(hidden_channels=64, metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
data = data.to(device)

In [10]:

os.makedirs('checkpoints', exist_ok=True)
num_epochs = 2
for epoch in range(num_epochs):
    loss = train(model, data, optimizer, device)
    print(f'Epoch {epoch+1:03d}, Loss: {loss:.4f}')
    
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch+1:03d}.pt')

print("Training complete! Models saved in 'checkpoints' directory.")

100%|██████████| 27087/27087 [27:46<00:00, 16.25it/s]


Epoch 001, Loss: 0.0860


100%|██████████| 27087/27087 [27:28<00:00, 16.43it/s]

Epoch 002, Loss: 0.0744
Training complete! Models saved in 'checkpoints' directory.


In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader  # Changed to NeighborLoader
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from scipy.special import expit  # Stable sigmoid
import numpy as np
import os

def load_model(model, optimizer, checkpoint_path, device):
    """Load model and optimizer state from a checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    print(f"Loaded checkpoint from {checkpoint_path}, epoch {epoch}, loss: {loss:.4f}")
    return model, optimizer

def evaluate(model, data, device, batch_size=128):
    """Evaluate the model on chemical-disease edge prediction using batched processing."""
    model.eval()
    all_scores = []
    all_labels = []
    
    # Use LinkNeighborLoader for batched evaluation (edge-centric)
    loader = LinkNeighborLoader(
        data,
        num_neighbors=[30,10],
        batch_size=batch_size,
        edge_label_index=(('chemical', 'chem_disease', 'disease'), data['chemical', 'chem_disease', 'disease'].edge_index),
        shuffle=False
    )
    
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            drug_emb = out['chemical']
            disease_emb = out['disease']
            
            pos_edge_index = batch['chemical', 'chem_disease', 'disease'].edge_index
            if pos_edge_index.size(1) == 0:
                continue
            pos_scores = predict_edges(drug_emb, disease_emb, pos_edge_index)
            pos_labels = torch.ones(pos_edge_index.size(1), device=device)
            
            neg_edge_index = negative_sampling(
                edge_index=pos_edge_index,
                num_nodes=(batch['chemical'].num_nodes, batch['disease'].num_nodes),
                num_neg_samples=pos_edge_index.size(1)
            )
            neg_scores = predict_edges(drug_emb, disease_emb, neg_edge_index)
            neg_labels = torch.zeros(neg_edge_index.size(1), device=device)
            
            scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
            labels = torch.cat([pos_labels, neg_labels]).cpu().numpy()
            all_scores.append(scores)
            all_labels.append(labels)
    
    all_scores = np.concatenate(all_scores)
    all_labels = np.concatenate(all_labels)
    
    # Stable sigmoid for probabilities
    probs = expit(all_scores)  # Replaces manual sigmoid to avoid overflow
    predictions = (probs > 0.5).astype(int)
    
    # Compute metrics, including balanced accuracy
    metrics = {
        'roc_auc': roc_auc_score(all_labels, probs),
        'precision': precision_score(all_labels, predictions, zero_division=0),
        'recall': recall_score(all_labels, predictions, zero_division=0),
        'f1': f1_score(all_labels, predictions, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(all_labels, predictions)
    }
    return metrics


In [11]:

def predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=5, device='cpu', batch_size=128):
    """Predict top-k drugs for a given disease ID using batched processing."""
    model.eval()
    if disease_id not in disease_idx:
        raise ValueError(f"Disease ID {disease_id} not found in disease_idx.")
    
    disease_global_idx = disease_idx[disease_id]
    all_scores = []
    all_chem_indices = []
    
    # Use NeighborLoader for node-centric batching on chemicals
    loader = NeighborLoader(
        data,
        num_neighbors=[30, 10],
        batch_size=batch_size,
        input_nodes=('chemical', torch.arange(data['chemical'].num_nodes)),  # All chemicals
        shuffle=False
    )
    
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            drug_emb = out['chemical']
            
            # Check if disease is in the batch subgraph (via node_id)
            if 'disease' in batch and disease_global_idx in batch['disease'].node_id:
                # Map global disease idx to batch-local idx
                local_disease_idx = (batch['disease'].node_id == disease_global_idx).nonzero(as_tuple=True)[0]
                if local_disease_idx.numel() > 0:
                    disease_emb_single = out['disease'][local_disease_idx].unsqueeze(0)
                    scores = (drug_emb * disease_emb_single).sum(dim=1).cpu().numpy()
                    batch_chem_globals = batch['chemical'].node_id.cpu().numpy()
                    all_scores.extend(scores)
                    all_chem_indices.extend(batch_chem_globals)
            
            # If disease not in batch, skip (or handle rarely with full forward if needed)
    
    if len(all_scores) == 0:
        raise ValueError(f"Disease {disease_id} not sampled in any batch. Increase num_neighbors or use full graph on CPU.")
    
    # Get top-k
    top_k_indices = np.argsort(all_scores)[::-1][:top_k]
    top_k_scores = [all_scores[i] for i in top_k_indices]
    top_k_chem_globals = [all_chem_indices[i] for i in top_k_indices]
    
    idx_to_chem = {i: c for c, i in chem_idx.items()}
    top_k_drugs = [(idx_to_chem[idx], score) for idx, score in zip(top_k_chem_globals, top_k_scores)]
    
    return top_k_drugs


In [ ]:

# Example usage (with CPU fallback)
device = torch.device('cpu' if torch.cuda.is_available() else 'cpu')
model = HeteroGNN(hidden_channels=64, metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

checkpoint_path = 'checkpoints/model_epoch_002.pt'
model, optimizer = load_model(model, optimizer, checkpoint_path, device)

metrics = evaluate(model, data, device, batch_size=128)
print("Evaluation Metrics:")
print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")


Loaded checkpoint from checkpoints/model_epoch_002.pt, epoch 2, loss: 0.0744


 69%|██████▉   | 18679/27087 [1:12:37<39:51,  3.52it/s]  